In [39]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/nyc311_features.csv", parse_dates=['created_date', 'closed_date'])
print(df.shape)
df.head()

(69321, 22)


,unique_key,created_date,closed_date,complaint_type,response_hours,response_hours_capped,response_hours_log,is_resolved,day_of_week,day_of_week_num,...,month,season,priority,module,borough,latitude,longitude,zip_code,status,agency
0,63572271,2024-12-31 23:56:00,2025-01-02 19:38:00,Noise,43.700000,43.700000,3.799974,1,Tuesday,1,...,12,Winter,Medium,Environment,Manhattan,40.779622,-73.975988,10023,Closed,DEP
1,63578403,2024-12-31 23:53:00,2025-01-01 09:15:00,Sewer,9.366667,9.366667,2.338596,1,Tuesday,1,...,12,Winter,Low,Water,Brooklyn,40.625472,-73.918895,11234,Closed,DEP
2,63582757,2024-12-31 23:50:55,2024-12-31 23:56:13,Animal in a Park,0.088333,0.088333,0.084647,1,Tuesday,1,...,12,Winter,Low,Parks,Staten Island,40.599974,-74.162847,10314,Closed,DPR
3,63580613,2024-12-31 23:50:00,2025-01-02 13:33:00,Street Light Condition,37.716667,37.716667,3.656270,1,Tuesday,1,...,12,Winter,Low,Electricity,Queens,40.678583,-73.842153,11417,Closed,DOT
4,63578347,2024-12-31 23:47:27,2025-01-02 12:55:35,Dirty Condition,37.135556,37.135556,3.641147,1,Tuesday,1,...,12,Winter,Low,Sanitation,Queens,40.692236,-73.865859,11421,Closed,DSNY


In [40]:
model_df = df[df['is_resolved'] == 1].copy()
print("Rows for modeling:", model_df.shape)
print("Nulls in target:", model_df['response_hours_log'].isnull().sum())

Rows for modeling: (67662, 22)
Nulls in target: 0


this model can only ever predict response time for complaints that would get resolved. It implicitly assumes resolution will happen — it cannot predict "this will never be resolved"

In [41]:
feature_cols = [
    'complaint_type', 'module', 'borough', 'priority', 'season',
    'day_of_week_num', 'is_weekend', 'hour_of_day', 'month'
]

target_col = 'response_hours_log'

X = model_df[feature_cols].copy()
y = model_df[target_col].copy()

print(X.shape, y.shape)
X.head()

(67662, 9) (67662,)


,complaint_type,module,borough,priority,season,day_of_week_num,is_weekend,hour_of_day,month
0,Noise,Environment,Manhattan,Medium,Winter,1,0,23,12
1,Sewer,Water,Brooklyn,Low,Winter,1,0,23,12
2,Animal in a Park,Parks,Staten Island,Low,Winter,1,0,23,12
3,Street Light Condition,Electricity,Queens,Low,Winter,1,0,23,12
4,Dirty Condition,Sanitation,Queens,Low,Winter,1,0,23,12


In [42]:
from sklearn.model_selection import train_test_split

# First split: 80% train+val, 20% test
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Second split: 75% train, 25% val (of the 80%) → 60% train, 20% val overall
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42
)

print("Train:", X_train.shape)
print("Val:", X_val.shape)
print("Test:", X_test.shape)

Train: (40596, 9)
Val: (13533, 9)
Test: (13533, 9)


In [43]:
# Compute mean response_hours_log per complaint_type, using TRAINING DATA ONLY
train_with_target = X_train.copy()
train_with_target['target'] = y_train

complaint_type_means = train_with_target.groupby('complaint_type')['target'].mean()

# Global fallback for complaint types seen in val/test but NOT in train
global_mean = y_train.mean()

print("Number of complaint types in training set:", len(complaint_type_means))
print("Global mean (fallback):", global_mean)

Number of complaint types in training set: 27
Global mean (fallback): 3.7202485940992824


In [44]:
X_train_te = X_train.copy()
X_val_te = X_val.copy()
X_test_te = X_test.copy()

X_train_te['complaint_type_encoded'] = X_train_te['complaint_type'].map(complaint_type_means)
X_val_te['complaint_type_encoded'] = X_val_te['complaint_type'].map(complaint_type_means).fillna(global_mean)
X_test_te['complaint_type_encoded'] = X_test_te['complaint_type'].map(complaint_type_means).fillna(global_mean)

# Drop the original text column now that we have the numeric encoding
X_train_te = X_train_te.drop(columns=['complaint_type'])
X_val_te = X_val_te.drop(columns=['complaint_type'])
X_test_te = X_test_te.drop(columns=['complaint_type'])

print(X_train_te[['complaint_type_encoded']].describe())
print("Nulls in val encoding:", X_val_te['complaint_type_encoded'].isnull().sum())
print("Nulls in test encoding:", X_test_te['complaint_type_encoded'].isnull().sum())

       complaint_type_encoded
count            40596.000000
mean                 3.720249
std                  1.096745
min                  1.079995
25%                  2.572291
50%                  3.714516
75%                  4.300742
max                  7.655152
Nulls in val encoding: 0
Nulls in test encoding: 0


In [45]:
categorical_cols = ['module', 'borough', 'priority', 'season']

X_train_encoded = pd.get_dummies(X_train_te, columns=categorical_cols, drop_first=True)
X_val_encoded = pd.get_dummies(X_val_te, columns=categorical_cols, drop_first=True)
X_test_encoded = pd.get_dummies(X_test_te, columns=categorical_cols, drop_first=True)

X_val_encoded = X_val_encoded.reindex(columns=X_train_encoded.columns, fill_value=0)
X_test_encoded = X_test_encoded.reindex(columns=X_train_encoded.columns, fill_value=0)

print(X_train_encoded.shape, X_val_encoded.shape, X_test_encoded.shape)

(40596, 17) (13533, 17) (13533, 17)


In [46]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

baseline = LinearRegression()
baseline.fit(X_train_encoded, y_train)

val_preds = baseline.predict(X_val_encoded)

mae = mean_absolute_error(y_val, val_preds)
rmse = np.sqrt(mean_squared_error(y_val, val_preds))
r2 = r2_score(y_val, val_preds)

print(f"Baseline — MAE: {mae:.3f}, RMSE: {rmse:.3f}, R²: {r2:.3f}")

Baseline — MAE: 1.098, RMSE: 1.454, R²: 0.374


In [47]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train_encoded, y_train)

val_preds_rf = rf.predict(X_val_encoded)

mae_rf = mean_absolute_error(y_val, val_preds_rf)
rmse_rf = np.sqrt(mean_squared_error(y_val, val_preds_rf))
r2_rf = r2_score(y_val, val_preds_rf)

print(f"Random Forest — MAE: {mae_rf:.3f}, RMSE: {rmse_rf:.3f}, R²: {r2_rf:.3f}")

Random Forest — MAE: 1.003, RMSE: 1.383, R²: 0.433


In [48]:
importances = pd.Series(rf.feature_importances_, index=X_train_encoded.columns)
importances = importances.sort_values(ascending=False)

print(importances.head(10))

import plotly.express as px
fig = px.bar(
    importances.head(10).reset_index(),
    x=0, y='index', orientation='h',
    title='Top 10 Feature Importances (Random Forest)',
    labels={'0': 'Importance', 'index': 'Feature'}
)
fig.show()

complaint_type_encoded    0.784743
hour_of_day               0.072467
day_of_week_num           0.048622
borough_Brooklyn          0.019908
borough_Queens            0.015273
borough_Manhattan         0.011903
month                     0.010840
borough_Staten Island     0.009404
is_weekend                0.007750
season_Winter             0.006232
dtype: float64


In [49]:
test_preds = rf.predict(X_test_encoded)

mae_test = mean_absolute_error(y_test, test_preds)
rmse_test = np.sqrt(mean_squared_error(y_test, test_preds))
r2_test = r2_score(y_test, test_preds)

print(f"Final Test — MAE: {mae_test:.3f}, RMSE: {rmse_test:.3f}, R²: {r2_test:.3f}")

# Convert back to real hours for interpretability
actual_hours = np.expm1(y_test)
predicted_hours = np.expm1(test_preds)

mae_hours = mean_absolute_error(actual_hours, predicted_hours)
print(f"MAE in actual hours: {mae_hours:.1f} hours")

Final Test — MAE: 0.991, RMSE: 1.369, R²: 0.450
MAE in actual hours: 125.4 hours


In [50]:
import joblib

joblib.dump(rf, "../models/response_time_model.pkl")
joblib.dump(list(X_train_encoded.columns), "../models/model_columns.pkl")

print("Model and columns saved.")

Model and columns saved.
